# 00a — The Geometry of Thought
## Linear Algebra as the Language of Neural Computation

---

**The paradigm shift this notebook installs:**

Old frame: *"Neuron 47 fired 12 spikes during maintenance, neuron 103 fired 8 spikes..."*  
New frame: *"The population state vector moved from point A to point B in ℝⁿ, and the geometry of that trajectory predicts whether the subject remembered the letter."*

This is not a stylistic preference. It is the correct level of description for understanding computation. Systems neuroscience has known this since Georgopoulos (1986) showed that a population vector predicts movement direction better than any single neuron. The field has been catching up ever since.

**What you will own after this notebook:**
1. Why eigenvalues of A determine the fate of every neural trajectory
2. Why SVD (not PCA, not factor analysis) is the right decomposition for neural data
3. How to measure the distance between two subspaces using principal angles
4. The connection between all three and working memory failure

**What this is NOT:**
A review of matrix operations. If you need `np.dot` syntax, use a reference. This notebook is about *conceptual ownership* — predicting what will happen before running code.

---
**Before you run anything:** Read *Brunton & Kutz, Data-Driven Science, Ch. 1* (SVD) — 20 pages. You own the PDF (`databook.pdf`). Then come back.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from mpl_toolkits.mplot3d import Axes3D

rng = np.random.default_rng(42)
plt.rcParams.update({'font.size': 12, 'figure.dpi': 120})

---
## Part 1 — Vectors as Neural States

Suppose you have 3 electrodes recording simultaneously. At any moment, you have 3 numbers: `[v1, v2, v3]`. This is a point in ℝ³.

Over 500 ms (600 samples at 1200 Hz), you get 600 such points — a **trajectory** through ℝ³.

With 40 electrodes, the trajectory is in ℝ⁴⁰. Same idea, harder to visualize.

**The claim:** the geometry of that trajectory — how it curves, how stable it is, which directions it explores — encodes the computation. Not the individual values. The trajectory geometry.

Let's build intuition on 3D first.

In [ ]:
# Generate three synthetic "neural states" — a 2D manifold embedded in 3D
# Think of this as 3 electrodes, 300 time points, 2 task conditions

T = 300  # time points (250 ms at 1200 Hz)
t = np.linspace(0, 2 * np.pi, T)

# Condition A: maintains item '1' — circular orbit in the (x,y) plane
condA = np.stack([np.cos(t), np.sin(t), 0.1 * rng.standard_normal(T)], axis=1)  # (T, 3)

# Condition B: maintains item '2' — orbit shifted and tilted
condB = np.stack([np.cos(t) * 0.5, np.sin(t) * 0.5 + 1.5, 0.3 * np.cos(t * 2)], axis=1)

# Condition C: failure trial — orbit collapses and drifts
decay = np.exp(-t / (2 * np.pi))
condC = np.stack([np.cos(t) * decay, np.sin(t) * decay, 0.2 * t / (2*np.pi)], axis=1)

fig = plt.figure(figsize=(14, 5))
for i, (cond, label, color) in enumerate([
    (condA, 'Correct (item 1)', 'royalblue'),
    (condB, 'Correct (item 2)', 'seagreen'),
    (condC, 'Failure', 'crimson')
]):
    ax = fig.add_subplot(1, 3, i+1, projection='3d')
    ax.plot(*cond.T, color=color, lw=1.5)
    ax.scatter(*cond[0], color=color, s=60, marker='o', label='start')
    ax.scatter(*cond[-1], color=color, s=60, marker='X', label='end')
    ax.set_title(label, fontsize=11)
    ax.set_xlabel('Electrode 1'); ax.set_ylabel('Electrode 2'); ax.set_zlabel('Electrode 3')

plt.suptitle('Neural Population Trajectories (Toy — 3 Electrodes)', fontweight='bold')
plt.tight_layout()
plt.show()

print("Notice: correct trials trace stable orbits. Failure = the orbit collapses.")
print("The SHAPE of the trajectory IS the computation. Not the individual values.")

### ✏️ Exercise 1.1 — Think Before Running

**Without running any new code, answer these:**

1. If you added Gaussian noise with σ=0.5 to condA, what would happen to the trajectory shape? Would it still look like a circle? What does this tell you about the *robustness* of the representation?

2. condA and condB both trace stable orbits. What geometric property separates them? (Hint: think about which direction in ℝ³ they differ most.)

3. If you were designing a decoder that distinguishes condA from condB, what would it compute? What if the only observable was electrode 1 (`x` coordinate)?

*Write your answers as comments below, then verify by plotting.*

In [ ]:
# Your verification code here
# Example: add noise to condA and plot

noisy_condA = condA + rng.standard_normal(condA.shape) * ???  # fill in sigma
# Plot and see

---
## Part 2 — The Dynamics Operator: What A Does

The simplest model of neural dynamics is a **linear dynamical system**:

$$x_{t+1} = A x_t + \text{noise}$$

where $x_t \in \mathbb{R}^n$ is the population state at time $t$ and $A \in \mathbb{R}^{n \times n}$ is the **transition matrix** — it encodes how the state evolves from one time step to the next.

**The key question:** Given $A$, what happens to the trajectory?

The answer is *completely* determined by the **eigenvalues** of $A$.

### Eigenvalue Intuition

An eigenvector $v$ of $A$ satisfies $Av = \lambda v$.

This means: if the state happens to lie along $v$, it just gets scaled by $\lambda$ at each step.

| Eigenvalue | Behavior | Neural interpretation |
|---|---|---|
| $|\lambda| < 1$ | Decays to zero | State is forgotten (dissipation) |
| $|\lambda| = 1$ | Stays constant | Persistent state — working memory! |
| $|\lambda| > 1$ | Grows without bound | Unstable — seizure-like activity |
| $\lambda$ complex | Rotates | Oscillation (theta, gamma) |

**The working memory attractor requires eigenvalues near the unit circle.** Too far inside = state decays (forgetting). Too far outside = runaway activity. The brain maintains memory by keeping eigenvalues at $|\lambda| \approx 1$.

This is WHY working memory is fragile. The system operates near a bifurcation point. A small perturbation (noise, distraction) can push eigenvalues inside the unit circle and collapse the state.

In [ ]:
# ─── Demonstrate eigenvalue behavior ───────────────────────────────────────

def simulate_linear_ds(A, x0, T=200, noise_std=0.05):
    """Simulate x_{t+1} = A x_t + noise."""
    x = np.zeros((T, len(x0)))
    x[0] = x0
    for t in range(T - 1):
        x[t+1] = A @ x[t] + rng.standard_normal(x0.shape) * noise_std
    return x

# Build three 2D systems with different eigenvalue regimes
# Stable (WM-like): rotation + slight decay
theta = 0.15  # rotation angle
A_wm = 0.98 * np.array([[np.cos(theta), -np.sin(theta)],
                          [np.sin(theta),  np.cos(theta)]])  # |λ| = 0.98 → slow decay

# Unstable: rotation + slight growth
A_unstable = 1.02 * np.array([[np.cos(theta), -np.sin(theta)],
                                [np.sin(theta),  np.cos(theta)]])  # |λ| = 1.02 → grows

# Collapsed: strong decay
A_collapsed = 0.7 * np.eye(2)  # |λ| = 0.7 → fast decay

x0 = np.array([1.0, 0.0])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
configs = [
    (A_wm,       'WM maintenance\n|λ|=0.98', 'royalblue'),
    (A_unstable, 'Unstable\n|λ|=1.02',       'crimson'),
    (A_collapsed,'Collapsed\n|λ|=0.70',       'gray'),
]

for ax, (A, title, color) in zip(axes, configs):
    traj = simulate_linear_ds(A, x0, T=200, noise_std=0.02)
    eigs = np.linalg.eigvals(A)
    
    ax.plot(traj[:, 0], traj[:, 1], color=color, lw=1.5, alpha=0.8)
    ax.scatter(*traj[0], color=color, s=80, zorder=5, label='start')
    # Draw unit circle for reference
    theta_ref = np.linspace(0, 2*np.pi, 100)
    ax.plot(np.cos(theta_ref), np.sin(theta_ref), 'k--', alpha=0.3, lw=1)
    ax.set_xlim(-2.5, 2.5); ax.set_ylim(-2.5, 2.5)
    ax.set_aspect('equal')
    ax.set_title(f'{title}\nλ = {eigs[0]:.3f}, {eigs[1]:.3f}', fontsize=10)
    ax.axhline(0, color='k', lw=0.5); ax.axvline(0, color='k', lw=0.5)

plt.suptitle('Linear Dynamical Systems — Three Eigenvalue Regimes', fontweight='bold')
plt.tight_layout()
plt.show()

### ✏️ Exercise 2.1 — Eigenvalue Computation by Hand

Given the 2×2 matrix:
$$A = \begin{pmatrix} 0.9 & -0.2 \\ 0.2 & 0.9 \end{pmatrix}$$

**By hand (no numpy):**
1. Compute $\det(A - \lambda I) = 0$ to find the characteristic polynomial
2. Solve for $\lambda$ — are they real or complex? What does this tell you about the trajectory?
3. Compute $|\lambda|$ — will this system decay, grow, or maintain?

**Then verify with numpy below.** If your hand answer doesn't match, find the error — don't just accept numpy's answer.

In [ ]:
A_ex = np.array([[0.9, -0.2], [0.2, 0.9]])

# Check your hand calculation:
eigs = np.linalg.eigvals(A_ex)
print('Eigenvalues:', eigs)
print('Magnitudes: ', np.abs(eigs))
print('Angles (Hz at 1200 srate):', np.angle(eigs) / (2 * np.pi) * 1200, 'Hz')

# Now simulate the system and see if it matches your prediction
traj = simulate_linear_ds(A_ex, np.array([1.0, 0.0]), T=400, noise_std=0.01)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot(traj[:, 0], traj[:, 1], 'royalblue', lw=1)
ax1.set_title('State space trajectory')
ax1.set_aspect('equal')

ax2.plot(traj)
ax2.set_xlabel('Time (samples)')
ax2.set_ylabel('State')
ax2.set_title('Time series')

plt.tight_layout()
plt.show()

### ✏️ Exercise 2.2 — The Fragility of WM

Working memory requires $|\lambda| \approx 1$. But biology is noisy.

**Experiment:** Start with `A_wm` from above. Add a small perturbation $\delta A = \epsilon \cdot$ random matrix with $\epsilon = 0.1$. Compute the new eigenvalues.

1. How often does the perturbation push $|\lambda|$ above 1 (unstable) vs. below 0.95 (fast decay)?
2. Run 100 perturbations. Plot the distribution of $|\lambda_{\max}|$.
3. **Conceptual question:** This is a toy model, but it captures something real. What biological mechanisms could introduce $\delta A$-like perturbations to prefrontal dynamics during a WM task?

*Write your answer to question 3 in a markdown cell. Then write the note `eigenvalues_and_stability.md` in `notes/`.*

In [ ]:
# Run 100 perturbations and plot |λ_max| distribution
eps = 0.1
max_eigs = []
for _ in range(100):
    dA   = rng.standard_normal((2, 2)) * eps
    Apert = A_wm + dA
    eigs = np.linalg.eigvals(Apert)
    max_eigs.append(np.abs(eigs).max())

plt.figure(figsize=(7, 4))
plt.hist(max_eigs, bins=30, color='royalblue', edgecolor='white')
plt.axvline(1.0, color='crimson', lw=2, label='Unit circle (stability boundary)')
plt.axvline(0.95, color='orange', lw=2, linestyle='--', label='|λ|=0.95 (fast decay)')
plt.xlabel('|λ_max| after perturbation')
plt.ylabel('Count')
plt.title('Eigenvalue fragility under random perturbations (ε=0.1)')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Fraction unstable (|λ|>1.0): {np.mean(np.array(max_eigs)>1.0):.2f}')
print(f'Fraction fast-decay (|λ|<0.95): {np.mean(np.array(max_eigs)<0.95):.2f}')

---
## Part 3 — SVD: The Natural Coordinate System for Neural Data

You have 40 electrodes. But the *effective* dimensionality of neural activity is much lower — typically 3-15 for a cognitive task. Why? Because neurons are not independent. They are coupled by synapses, shared inputs, and common modulatory signals. This coupling creates **correlated structure** that lives in a low-dimensional subspace.

**The Singular Value Decomposition (SVD) finds this subspace.**

$$X = U \Sigma V^\top$$

Where $X \in \mathbb{R}^{T \times C}$ (T timepoints, C channels):

| Matrix | Shape | Meaning |
|--------|-------|---------|
| $U$ | $T \times T$ | Temporal modes — how each pattern evolves in time |
| $\Sigma$ | $T \times C$ | Singular values — how much variance each mode captures |
| $V$ | $C \times C$ | Spatial modes — which electrodes participate in each pattern |

**The key insight:** The columns of $V$ (or equivalently, the rows of $V^\top$) are the **principal directions** of the data. They are the axes of an ellipsoid that describes the spread of population activity. The singular values tell you the length of each axis.

**For neural data:** the first few singular vectors capture the bulk of the variance. Everything else is noise. You project your data onto these to get a clean low-dimensional trajectory.

**Why SVD and not PCA?** They are equivalent. PCA = SVD after centering. The distinction matters only for the implementation — SVD is numerically more stable on tall-thin matrices.

In [ ]:
# ─── SVD on synthetic neural data ─────────────────────────────────────────
# Simulate: 50 channels, 1000 time points, 3D manifold + noise

n_channels = 50
n_times    = 1000
n_latent   = 3   # true dimensionality

# Low-dimensional latent dynamics (3D)
t_lin  = np.linspace(0, 4 * np.pi, n_times)
latent = np.stack([
    np.sin(t_lin),
    np.cos(t_lin) * np.sin(t_lin * 0.5),
    0.3 * t_lin / (4 * np.pi)
], axis=1)  # (T, 3)

# Mix latent into 50 channels via random mixing matrix
W = rng.standard_normal((n_latent, n_channels)) * 2
X_clean = latent @ W                            # (T, 50) — the signal
X_noisy = X_clean + rng.standard_normal(X_clean.shape) * 3  # add noise

# Apply SVD
X_c = X_noisy - X_noisy.mean(axis=0)           # center
U, s, Vt = np.linalg.svd(X_c, full_matrices=False)

# Variance explained per component
var_exp = (s**2) / (s**2).sum()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1: Singular value spectrum
axes[0].bar(range(1, 21), var_exp[:20] * 100, color='steelblue')
axes[0].axvline(3.5, color='crimson', lw=2, linestyle='--', label='True dim=3')
axes[0].set_xlabel('Component')
axes[0].set_ylabel('Variance explained (%)')
axes[0].set_title('Singular Value Spectrum\n(scree plot)')
axes[0].legend()

# 2: Projection onto top 2 PCs — noisy vs clean
scores = U[:, :3] * s[:3]   # = X_c @ Vt[:3].T, top 3 PCs
axes[1].plot(scores[:, 0], scores[:, 1], color='steelblue', lw=1, label='Projected')
axes[1].set_xlabel('PC 1'); axes[1].set_ylabel('PC 2')
axes[1].set_title('Top 2 PCs recover\nthe latent manifold')

# 3: Reconstruction quality with different # components
for k in [1, 3, 10, 50]:
    X_recon = (U[:, :k] * s[:k]) @ Vt[:k]
    recon_err = np.mean((X_c - X_recon)**2) / X_c.var()
    axes[2].scatter(k, recon_err, s=80, label=f'k={k}: err={recon_err:.3f}')

axes[2].set_xlabel('# Components used')
axes[2].set_ylabel('Normalized reconstruction error')
axes[2].set_title('How many components\ndo you need?')
axes[2].legend(fontsize=9)
axes[2].set_xscale('log')

plt.suptitle('SVD on Synthetic Neural Data (50 channels, 3D manifold)', fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Cumulative variance explained by top 3 PCs: {var_exp[:3].sum()*100:.1f}%')
print(f'Cumulative variance explained by top 10 PCs: {var_exp[:10].sum()*100:.1f}%')

### ✏️ Exercise 3.1 — Predict the Scree Plot

**Before running code, predict:**

1. If you increase the noise from σ=3 to σ=10, what happens to the scree plot? Does the "elbow" shift? Why?

2. If you add a 4th latent dimension to the simulation, where does the elbow move? What if the 4th dimension has much smaller variance than the first 3?

3. In real neural data, the scree plot rarely has a sharp elbow. What does a smooth, gradual scree plot tell you about the data structure?

4. **The hard one:** You run PCA on 40 electrodes and get 40 components. Your colleague runs PCA on the same 40 electrodes but uses only 300 (instead of 1000) time points. They get the same 40 components but the singular values are very different. Who has the better estimate of the true PC directions, and why?

*Write answers as comments, then verify experimentally.*

In [ ]:
# Experiment with Exercise 3.1
# Vary noise level and see what happens to the scree plot

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, noise_std in zip(axes, [1.0, 3.0, 10.0]):
    X_n = X_clean + rng.standard_normal(X_clean.shape) * noise_std
    X_n -= X_n.mean(axis=0)
    _, s_n, _ = np.linalg.svd(X_n, full_matrices=False)
    var_n = (s_n**2) / (s_n**2).sum()
    ax.bar(range(1, 16), var_n[:15] * 100, color='steelblue')
    ax.axvline(3.5, color='crimson', lw=2, linestyle='--')
    ax.set_title(f'Noise σ={noise_std}')
    ax.set_xlabel('Component')
    ax.set_ylabel('Variance explained (%)')

plt.suptitle('Effect of noise on singular value spectrum', fontweight='bold')
plt.tight_layout()
plt.show()

---
## Part 4 — Principal Angles: Distance Between Subspaces

In the anchor project, the core claim is:

> *During successful WM maintenance, neural population activity for different items lives in **distinct** subspaces. Before failure, these subspaces **collapse** toward each other — representational interference.*

To test this, we need to measure the **distance between two subspaces**. Not between two points — between two entire subspaces.

**Principal angles** are the answer.

### The Math

Given two subspaces $\mathcal{S}_1$ and $\mathcal{S}_2$ spanned by matrices $A$ and $B$:

1. Orthonormalize: compute $Q_A, Q_B$ (QR decomposition)
2. Compute $M = Q_A^\top Q_B$
3. SVD: $M = U \Sigma V^\top$
4. Principal angles: $\theta_i = \arccos(\sigma_i)$

**Why this works geometrically:** $\sigma_i$ is the cosine of the angle between the $i$-th "most aligned" pair of directions. If $\sigma_1 = 1$, the subspaces share a direction (overlap). If all $\sigma_i \approx 0$, they are orthogonal.

**Interpretation:**
- $\theta_{\min}$ small (≈0°) → subspaces nearly identical → WM items interfere → failure
- $\theta_{\min}$ large (≈90°) → subspaces orthogonal → items cleanly separated → success

In [ ]:
# ─── Principal angles demo ─────────────────────────────────────────────────

def principal_angles(A, B):
    """Compute principal angles between subspaces spanned by cols of A and B."""
    Qa, _ = np.linalg.qr(A)
    Qb, _ = np.linalg.qr(B)
    _, sv, _ = np.linalg.svd(Qa.T @ Qb, full_matrices=False)
    sv = np.clip(sv, -1, 1)
    return np.arccos(sv) * 180 / np.pi  # return in degrees

# Case 1: Identical subspaces → angles should all be 0
A = rng.standard_normal((10, 3))  # 10D ambient, 3D subspace
angles_identical = principal_angles(A, A)

# Case 2: Orthogonal subspaces → angles should all be 90°
# Build B orthogonal to A using QR
full, _ = np.linalg.qr(rng.standard_normal((10, 10)))
B_orth = full[:, 3:6]  # columns orthogonal to full[:, :3]
angles_orth = principal_angles(A, B_orth)

# Case 3: Partial overlap
B_partial = A + 0.5 * rng.standard_normal(A.shape)
angles_partial = principal_angles(A, B_partial)

print("Identical subspaces:  θ =", np.round(angles_identical, 1))
print("Orthogonal subspaces: θ =", np.round(angles_orth, 1))
print("Partial overlap:      θ =", np.round(angles_partial, 1))

In [ ]:
# ─── Simulate WM maintenance and failure using principal angles ─────────────

# Simulate two WM items as trajectories in a shared 10D space
n_ambient = 10
n_dims    = 3  # subspace dimension per item
T_trial   = 50  # time points in maintenance

def make_maintenance_traj(center, noise=0.1):
    """Simulate maintenance period: trajectory stays near 'center' direction."""
    t = np.linspace(0, 2*np.pi, T_trial)
    # Stable orbit in the subspace defined by center + random transverse
    return center.reshape(1,-1) * np.cos(t).reshape(-1,1) + rng.standard_normal((T_trial, n_ambient)) * noise

# Two items with orthogonal centers (successful maintenance)
c1 = np.zeros(n_ambient); c1[:3] = [1, 0, 0]
c2 = np.zeros(n_ambient); c2[3:6] = [0, 1, 0]  # orthogonal to c1

# Collapse scenario: c2 drifts toward c1 over time
n_bins   = 20   # time bins in trial
mix_vals = np.linspace(0, 1, n_bins)  # mixing coefficient: 0=separate, 1=identical

theta_min_success = []
theta_min_failure = []

for alpha in mix_vals:
    # Success: c2 stays orthogonal
    traj1 = make_maintenance_traj(c1, noise=0.05)
    traj2 = make_maintenance_traj(c2, noise=0.05)
    ang_s = principal_angles(traj1, traj2)
    theta_min_success.append(ang_s.min())
    
    # Failure: c2 drifts toward c1
    c2_drift = (1 - alpha) * c2 + alpha * c1
    c2_drift /= np.linalg.norm(c2_drift) + 1e-8
    traj2_f = make_maintenance_traj(c2_drift, noise=0.05)
    ang_f = principal_angles(traj1, traj2_f)
    theta_min_failure.append(ang_f.min())

plt.figure(figsize=(8, 4))
plt.plot(mix_vals, theta_min_success, 'royalblue', lw=2, label='Success (items stay separate)')
plt.plot(mix_vals, theta_min_failure, 'crimson',   lw=2, label='Failure (item 2 drifts toward item 1)')
plt.xlabel('Time in trial (0=encoding, 1=pre-response)')
plt.ylabel('θ_min (degrees)')
plt.title('Principal Angle Collapse Predicts WM Failure\n(Toy simulation)')
plt.legend()
plt.axhline(0, color='k', lw=0.5, linestyle='--')
plt.tight_layout()
plt.show()

print("The geometric prediction: θ_min collapse is the early warning signal.")
print("This is what we'll compute on real Miller data in Module 3.")

### ✏️ Exercise 4.1 — The Mastery Test

**Answer these without running new code. Verify afterward.**

1. In the principal angle simulation above, what happens if you increase noise from 0.05 to 0.5? Does θ_min increase or decrease for the success condition? Why? (Hint: think about what noise does to the subspace estimate.)

2. In real N-back data, we have 0-back, 1-back, and 2-back conditions. How would you expect θ_min between the 1-back and 2-back subspaces to compare to θ_min between 0-back and 2-back? Why?

3. You compute θ_min for a subject and find it's 87° throughout the maintenance period on both correct and incorrect trials. The patient's WM accuracy is 90%. Your colleague says "the geometry doesn't distinguish correct from incorrect." What alternative interpretations should you consider before accepting this conclusion?

4. **Design question:** You have 2-back data. How would you define the two subspaces to compute a meaningful θ_min? (There's no single right answer — write out your reasoning.)

---
## Mastery Checkpoint

You own this material when you can answer all of these without reference:

**Eigenvalues:**
- [ ] Given a 2×2 A matrix, compute its eigenvalues by hand and predict the trajectory shape
- [ ] Explain why WM requires eigenvalues near the unit circle in terms of the biology
- [ ] Know when linear models fail (non-Gaussian noise, switching dynamics, non-stationarity)

**SVD:**
- [ ] Interpret the scree plot in terms of the data's intrinsic dimensionality
- [ ] Explain why the first singular vector captures "the most variance" geometrically
- [ ] Predict how the scree plot changes with more noise or fewer observations

**Principal Angles:**
- [ ] Explain why low θ_min signals representational interference
- [ ] Design a subspace comparison that would test a specific hypothesis about WM load
- [ ] Know why you need QR decomposition before computing the inner product of subspaces

---
## Next Step

Write `notes/svd_neural_data.md` — derive SVD's connection to PCA from scratch.  
Then go to `01_data_signal/01_load_explore_ieeg.ipynb` — apply everything to real Miller data.